# LLM Finetuning Sprint

In [1]:
# 1. Clone the repository into the Colab environment
!git clone https://github.com/MircoFernando/Advanced-Hybrid-RAG-Finacial-Intelligence.git

fatal: destination path 'Advanced-Hybrid-RAG-Finacial-Intelligence' already exists and is not an empty directory.


In [1]:
%cd Advanced-Hybrid-RAG-Finacial-Intelligence

/content/Advanced-Hybrid-RAG-Finacial-Intelligence


In [2]:
!git checkout data-factory

Already on 'data-factory'
Your branch is up to date with 'origin/data-factory'.


## 1. Installing Dependencies

In [3]:
%%capture

# Core training libraries
!pip install -q \
    transformers==4.44.2 \
    datasets==2.20.0 \
    tokenizers==0.19.1 \
    accelerate==0.34.2 \
    peft==0.12.0 \
    trl==0.9.6 \
    bitsandbytes==0.46.1 \
    evaluate==0.4.2

# Utilities
!pip install -q \
    numpy \
    pandas \
    scikit-learn \
    rich \
    pyyaml \
    python-dotenv \
    tqdm

# Evaluation
!pip install -q pydantic>=2.9 openai rouge-score

print("✅ Installation complete!")
print("✅ All dependencies compatible (pydantic v2 + openai)")

In [5]:
print("🔧 Installing unsloth...")
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q

🔧 Installing unsloth...
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [4]:
import numpy as np
print(np.__version__)

1.26.4


In [7]:
!pip install --upgrade numpy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 81.8 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
trl 0.9.6 requires numpy<2.0.0,>=1.18.2, but you have numpy 2.4.6 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.6 which is incompatible.


## 2. Environment & GPU Check

In [6]:
import sys
import torch

print("="*60)
print("ENVIRONMENT CHECK")
print("="*60)
print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"Device name: {torch.cuda.get_device_name(0)}")
    print(f"Device capability: {torch.cuda.get_device_capability(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️ WARNING: CUDA not available. Training will be VERY slow on CPU.")

print("="*60)

ENVIRONMENT CHECK
Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA version: 12.8
Device name: Tesla T4
Device capability: (7, 5)
Total VRAM: 14.56 GB


## 3. Seeds & Determinism

In [7]:
import os
import random
import numpy as np
import torch

SEED = 42

# Set environment variable for Python hash seed
os.environ['PYTHONHASHSEED'] = str(SEED)

# Set seeds
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    # Note: These settings may impact performance
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f"✅ Seeds set to {SEED} for reproducibility")
print("⚠️ Note: Full determinism on GPU is not guaranteed due to non-deterministic operations")

✅ Seeds set to 42 for reproducibility
⚠️ Note: Full determinism on GPU is not guaranteed due to non-deterministic operations


## 4. Hugging Face Login

In [8]:
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
hf_token = userdata.get("HF_TOKEN")
!hf auth login --token $HF_TOKEN

print("ℹ️ Hugging Face login skipped. Uncomment login() to push models to Hub.")

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: write).
The token `Mirco` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
ℹ️ Hugging Face login skipped. Uncomment login() to push models to Hub.


## 5. Configuration

In [22]:
from pathlib import Path
from pprint import pprint

# Auto-detect compute dtype (BF16 requires compute capability >= 8.0)
use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

# Resolve the project root so the notebook works from either the repo root or the notebooks folder.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "artifacts").exists() and (PROJECT_ROOT.parent / "artifacts").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

UBER_DATASET_PATH = PROJECT_ROOT / "artifacts" / "data_factory" / "uber_dataset.json"
TRAIN_JSONL_PATH = PROJECT_ROOT / "artifacts" / "train" / "train.jsonl"
GOLDEN_TEST_SET_PATH = PROJECT_ROOT / "artifacts" / "test" / "golden_test_set.jsonl"

CONFIG = {
    # Model
    "base_model": "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",

    # Local Uber data
    "train_files": [str(UBER_DATASET_PATH), str(TRAIN_JSONL_PATH)],
    "eval_file": str(GOLDEN_TEST_SET_PATH),
    "dataset_sample": 1500,
    "train_val_split": 0.8,  # 90% train, 10% validation

    # Tokenization
    "max_length": 1024,

    # Training
    "num_train_epochs": 1,
    "max_steps": 200,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 16,
    "learning_rate": 2e-4,
    "warmup_ratio": 0.03,
    "logging_steps": 10,
    "save_steps": 50,
    "eval_steps": 50,
    "save_total_limit": 2,

    # LoRA
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "lora_target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],

    # Quantization
    "load_in_4bit": True,
    "bnb_4bit_compute_dtype": compute_dtype,
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_use_double_quant": True,

    # Output
    "output_dir": "outputs/adapter",
    "push_to_hub": False,

    # Generation
    "max_new_tokens": 256,
    "temperature": 0.0,
    "do_sample": False,
}

# Effective batch size
effective_batch_size = CONFIG["per_device_train_batch_size"] * CONFIG["gradient_accumulation_steps"]

print("="*60)
print("CONFIGURATION")
print("="*60)
pprint(CONFIG)
print("="*60)
print(f"Project root: {PROJECT_ROOT}")
print(f"Uber dataset: {UBER_DATASET_PATH}")
print(f"Train JSONL: {TRAIN_JSONL_PATH}")
print(f"Golden test set: {GOLDEN_TEST_SET_PATH}")
print(f"Compute dtype: {compute_dtype}")
print(f"Using BF16: {use_bf16}")
print(f"Effective batch size: {effective_batch_size}")
print("="*60)

CONFIGURATION
{'base_model': 'unsloth/Llama-3.2-3B-Instruct-bnb-4bit',
 'bnb_4bit_compute_dtype': torch.float16,
 'bnb_4bit_quant_type': 'nf4',
 'bnb_4bit_use_double_quant': True,
 'dataset_sample': 1500,
 'do_sample': False,
 'eval_file': '/content/Advanced-Hybrid-RAG-Finacial-Intelligence/artifacts/test/golden_test_set.jsonl',
 'eval_steps': 50,
 'gradient_accumulation_steps': 16,
 'learning_rate': 0.0002,
 'load_in_4bit': True,
 'logging_steps': 10,
 'lora_alpha': 32,
 'lora_dropout': 0.05,
 'lora_r': 16,
 'lora_target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj'],
 'max_length': 1024,
 'max_new_tokens': 256,
 'max_steps': 200,
 'num_train_epochs': 1,
 'output_dir': 'outputs/adapter',
 'per_device_train_batch_size': 1,
 'push_to_hub': False,
 'save_steps': 50,
 'save_total_limit': 2,
 'temperature': 0.0,
 'train_files': ['/content/Advanced-Hybrid-RAG-Finacial-Intelligence/artifacts/data_factory/uber_dataset.json',
                 '/content/Advanced-Hybrid-RAG-Finacial-Intellig

## 6. Dataset Loading

In [10]:
from datasets import load_dataset, Dataset

# Load the dataset
def load_uber_dataset(sample):

  dataset = load_dataset("json", data_files="/content/Advanced-Hybrid-RAG-Finacial-Intelligence/artifacts/train/train.jsonl", split="train")
  print(f"✅ Created synthetic dataset with {len(dataset)} examples")

  return dataset.select(range(sample))

dataset = load_uber_dataset(CONFIG["dataset_sample"])

original_dataset_len = len(dataset) # Store original length for dropped examples calculation

print(f"\n📊 Dataset before cleaning: {len(dataset)} examples")

# Drop rows with missing instruction or output
dataset = dataset.filter(lambda x: x["context"] is not None and x["answer"] is not None)

print(f"📊 Dataset after cleaning: {len(dataset)} examples")
print(f"✅ Dropped {original_dataset_len - len(dataset)} examples with missing data\n") # Updated print statement

# Split into train/validation
split_dataset = dataset.train_test_split(
    train_size=CONFIG["train_val_split"],
    seed=SEED
)
train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]

print(f"📊 Train: {len(train_dataset)} | Validation: {len(val_dataset)}")
print("\n📝 Sample example:")
print(dataset[0])

print("\n📝 Sample training example:")
print(train_dataset[0])

print("\n📝 Sample test example:")
print(val_dataset[0])


Generating train split: 0 examples [00:00, ? examples/s]

✅ Created synthetic dataset with 4336 examples

📊 Dataset before cleaning: 1500 examples


Filter:   0%|          | 0/1500 [00:00<?, ? examples/s]

📊 Dataset after cleaning: 1500 examples
✅ Dropped 0 examples with missing data

📊 Train: 1200 | Validation: 300

📝 Sample example:
{'context': '67 \nPerforming a quantitative goodwill impairment test includes the determination of the fair value of a reporting unit and involves \nsignificant estimates and assumptions. These estimates and assumptions include, among others, revenue growth rates and operating \nmargins used to calculate projected future cash flows, risk-adjusted discount rates, future economic and market conditions, and the \ndetermination of appropriate market comparables. \nLoss Contingencies \nWe are involved in legal proceedings, claims, and regulatory, indirect tax examinations, or government inquiries and \ninvestigations that may arise in the ordinary course of business. Certain of these matters include speculative claims for substantial or \nindeterminate amounts of damages. We record a liability when we believe that it is both probable that a loss has been incurre

### Preview First 50 samples

In [11]:
import pandas as pd

# Convert first 50 samples to dataframe
df_preview = pd.DataFrame(train_dataset[:50])

# Display with formatting
pd.set_option('display.max_colwidth', 100)  # Limit column width for readability
print(f"📊 Displaying first 50 samples out of {len(dataset)} total examples\n")
df_preview

📊 Displaying first 50 samples out of 1500 total examples



,context,question,answer
0,"Debt Securities \nAs of December 31, 2023, the amortized cost of our debt securities approximate...","What was the weighted-average remaining maturity of debt securities as of December 31, 2024?",Less than one year.
1,"51 \nfinancial statements included in Part II, Item 8, “Financial Statements and Supplementary D...",Who are the employees mentioned that support operations in cities?,"General managers, Driver operations, platform user support representatives, and community managers."
2,"2029 Senior Note \n \n1,450 \nTotal \n $ \n2,668 \nThe future principal payments for our long...",How much was recognized for the amortization of debt discount and issuance costs in 2023?,$18 million
3,"Drivers and Merchants to seek, receive and fulfill on-demand requests from end-users seeking Mob...",What does the Master Services Agreement (MSA) define regarding the service fee?,It defines the service fee Uber charges Drivers and Merchants for each transaction.
4,39 \nAdverse litigation judgments or settlements resulting from legal proceedings in which we ma...,How does the company describe the nature of the legal matters it is involved with?,They have arisen in the normal course of business and include various claims and matters.
5,that we are an agent in these arrangements as we arrange for other parties to provide the servic...,How is revenue represented in certain markets where we are responsible for Mobility or Delivery ...,"Revenue is presented on a gross basis from end-users and Shippers, with payments to Drivers and ..."
6,includes activity related to our financial partnerships products and advertising. \nDelivery \nO...,How many years ago was the Uber Eats app launched?,Over nine years ago.
7,"of all available evidence, whether it is more-likely-than-not that some or all of the deferred t...",What action does the company intend to continue regarding the valuation allowance on net deferre...,They will continue to maintain a valuation allowance against them.
8,"31 \nmergers, and sales of assets, and restrictions on the payment of dividends or distributions...",What potential financial liability is the company exposed to that could be materially greater th...,Tax liabilities
9,7 \nPayments and Financial Services \nMost jurisdictions in which we operate have laws that gove...,Which sectors of Uber's business could be affected by changes in laws governing payment processing?,Payment processing and financial services.


## 7. Loading the Model and Tokenizer using UnSloth

In [12]:
from transformers import BitsAndBytesConfig
import torch
from unsloth import FastLanguageModel

bnb_config = BitsAndBytesConfig(
    load_in_4bit=CONFIG["load_in_4bit"],
    bnb_4bit_compute_dtype=CONFIG["bnb_4bit_compute_dtype"],
    bnb_4bit_quant_type=CONFIG["bnb_4bit_quant_type"],
    bnb_4bit_use_double_quant=CONFIG["bnb_4bit_use_double_quant"],
)

print(f"📥 Loading base model: {CONFIG['base_model']}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
    quantization_config = bnb_config,
)

print("✅ Base model loaded in 4-bit")

/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:144: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
📥 Loading base model: unsloth/Llama-3.2-3B-Instruct-bnb-4bit...
==((====))==  Unsloth 2026.5.6: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.


✅ Base model loaded in 4-bit


## 8. Token Length & Truncation Diagnostics

In [15]:
import numpy as np

# Ensure pad token is set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Sample data
sample_size = min(1500, len(train_dataset))
sample_dataset = train_dataset.select(range(sample_size))

# Compute token lengths
token_lengths = []
for example in sample_dataset:
    text = f"{example['question']} {example['answer']} {example['context']}"
    # tokenizer utlizing
    tokens = tokenizer(text, add_special_tokens=True)
    token_lengths.append(len(tokens["input_ids"]))

token_lengths = np.array(token_lengths)

# --- Print results ---
print("="*60)
print("TOKEN LENGTH DIAGNOSTICS")
print("="*60)
print(f"Sample size: {sample_size}")
print(f"Average: {token_lengths.mean():.1f}")
print(f"95th percentile: {np.percentile(token_lengths, 95):.1f}")

truncated = (token_lengths > CONFIG["max_length"]).sum()
print(f"Truncation at {CONFIG['max_length']}: {truncated}/{len(token_lengths)}")
print("="*60)

TOKEN LENGTH DIAGNOSTICS
Sample size: 1200
Average: 287.1
95th percentile: 420.0
Truncation at 1024: 0/1200


## 9. Build SFT Prompts

In [23]:
def chatml_format(context=None, system_text = "Answer the question accurately using only the provided context from the Uber 2024 Annual Report.", question=None, answer=None):
    # System prompt specific to the Uber 2024 Annual Report task


   # Combining Context and Question into the User message
    user_text = f"Context:\n{context}\n\nQuestion:\n{question}"

    messages = [
        {"role": "system", "content": system_text},
        {"role": "user", "content": user_text},
    ]

    # If an answer exists (for training), append it as the Assistant role
    if answer:
        messages.append({"role": "assistant", "content": answer})

    # Let the Llama-3 Tokenizer generate the perfect Meta string
    # tokenize=False returns the raw text string, which SFTTrainer needs!
    formatted = tokenizer.apply_chat_template(messages, tokenize=False)

    return formatted

def build_sft_prompt(row):
    """
    Mapping function to apply ChatML formatting to each row of the dataset.
    Uses 'context', 'question', and 'answer' fields from the input row.
    """
    prompt = chatml_format(
        context = row["context"],
        question = row["question"],
        answer = row["answer"]
    )
    return {"text": prompt}

# Apply the mapping to the train and validation datasets
train_dataset = train_dataset.map(build_sft_prompt)
val_dataset = val_dataset.map(build_sft_prompt)

print("✅ Prompts built for SFT in ChatML format")
print("\n📝 Sample formatted prompt:")
print(train_dataset[0]["text"])

Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

✅ Prompts built for SFT in ChatML format

📝 Sample formatted prompt:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 23 May 2026

Answer the question accurately using only the provided context from the Uber 2024 Annual Report.<|eot_id|><|start_header_id|>user<|end_header_id|>

Context:
Debt Securities 
As of December 31, 2023, the amortized cost of our debt securities approximates fair value. We did not record any material 
unrealized gains or losses as of December 31, 2023. 
The following table summarizes the amortized cost, unrealized gains and losses, and fair value of our debt securities (in millions): 
  
 
As of December 31, 2024 
  
 
Amortized Cost 
 
Unrealized Gains  
Unrealized Losses  
Fair Value 
U.S. government and agency securities 
 $ 
5,843  $ 
7  $ 
(2) $ 
5,848  
Commercial paper 
  
702   
—   
—   
702  
Corporate bonds 
  
1,975   
1   
(2)  
1,974  
Certificates of deposit 
  
38   
—   
—   
38  
Tot

## 10. Baseline Inference (Before Finetuning)

In [24]:
import time
import torch

test_prompts = [
    {
        "title": "Uber Mission",
        "prompt": "What is Uber's mission statement and what does it emphasize?"
    },
    {
        "title": "Freight Segment",
        "prompt": "What does the Freight segment connect on the platform?"
    },
    {
        "title": "Board Member",
        "prompt": "Who amongst the board members is associated with Flex Ltd.?"
    },
    {
        "title": "Stock Symbol",
        "prompt": "What symbol is used for Uber Technologies, Inc.'s common stock on the New York Stock Exchange?"
    },
]


print("\n" + "="*60)
print("BASELINE OUTPUTS (PRE-FINETUNE)")
print("="*60)

if torch.cuda.is_available():
    vram_before = torch.cuda.memory_allocated() / 1024**3
    print(f"VRAM before generation: {vram_before:.2f} GB\n")

for i, test in enumerate(test_prompts, 1):
    # Format as ChatML
   # Format as ChatML using the updated function signatures
    formatted_prompt = chatml_format(question=test['prompt'], system_text="You are an expert assistant for the Uber 2024 Annual Report, Use only provided sources. If unknown reply: I don't know — please provide the source. Answer in one short sentence; cite source if available.")
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

    start_time = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=CONFIG["max_new_tokens"],
            temperature=CONFIG["temperature"] if CONFIG["temperature"] > 0 else None,
            do_sample=CONFIG["do_sample"],
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    elapsed = time.time() - start_time
    # Decode the output, slicing off the input prompt so we only see the AI's new words
    input_length = inputs["input_ids"].shape[1]
    generated_text = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
    print(f"AI: {generated_text.strip()}")

    # Calculate metrics
    tokens_generated = outputs.shape[1] - inputs["input_ids"].shape[1]
    tokens_per_sec = tokens_generated / elapsed if elapsed > 0 else 0

    print(f"\nLatency: {elapsed:.2f}s | Speed: {tokens_per_sec:.1f} tok/s")

if torch.cuda.is_available():
    vram_after = torch.cuda.memory_allocated() / 1024**3
    print(f"\n📊 VRAM after generation: {vram_after:.2f} GB")
    print(f"📊 VRAM delta (Cost of thinking): {vram_after - vram_before:.2f} GB")

print("\n" + "="*60)

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



BASELINE OUTPUTS (PRE-FINETUNE)
VRAM before generation: 2.23 GB



Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


AI: assistant

Uber's mission statement is "to make transportation as accessible as water" (Source: Uber's 2022 Annual Report, p. 4).

Latency: 1.46s | Speed: 23.9 tok/s


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


AI: assistant

The Freight segment connects shippers with carriers and logistics providers.

Latency: 0.74s | Speed: 22.9 tok/s


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


AI: assistant

I don't know.

Latency: 0.47s | Speed: 21.3 tok/s
AI: assistant

UBER.

Latency: 0.39s | Speed: 20.4 tok/s

📊 VRAM after generation: 2.23 GB
📊 VRAM delta (Cost of thinking): 0.00 GB

